# Plotly Workshop

This workshop introduces **Plotly**, a Python library for building interactive, publication-quality charts. We'll work through several common chart types — bar charts, histograms, scatter plots, line charts, and more — using real fire incident data from the FDNY.

## Setup

Run the cell below to import all required libraries before proceeding.

In [ ]:
import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np

import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

## Overview

We'll primarily work with **`plotly.express`** (`px`) — a high-level module for building common chart types quickly with minimal code. For more advanced visualizations, specifically Sankey diagrams, we'll use **`plotly.graph_objects`** (`go`), which provides lower-level control over figure construction.

The charts produced by Plotly are fully interactive: hover over data points to see values, zoom in and out, and click legend items to toggle series on and off — all directly within the notebook.

**Dataset:** Fire Incident Dispatch Data from the FDNY's STARFIRE system, covering incidents in New York City from August through December 2024.

## 1. Loading the Data

The dataset comes from the FDNY's STARFIRE dispatch system and contains nearly 300,000 fire incident records. Each row represents a single incident and includes location, timestamps, alarm level, and units assigned.

In [ ]:
#read in the data
df=pd.read_csv('https://raw.githubusercontent.com/nickxscott/INFO658/refs/heads/main/notebooks/data/Fire_Incident_Dispatch_Data_20260315.csv')
df.head()

In [ ]:
print(f'There are {len(df)} rows in this dataset')

Let's inspect the column names and data types to understand what we're working with before we start cleaning.

In [ ]:
df.columns

In [ ]:
df.dtypes

## 2. Data Cleaning

Before visualizing the data, we need to clean and prepare it. The cells in this section address data type issues, missing values, outliers, and the creation of derived variables.

Here's a summary of the cleaning steps we'll work through:

- Convert `INCIDENT_DATETIME`, `FIRST_ON_SCENE_DATETIME`, and `INCIDENT_CLOSE_DATETIME` to `datetime` type
- Calculate **time on scene** — the number of seconds between first arrival and incident close
- Extract a `date` column (date only, no timestamp) and a `month` column for grouping
- Convert `INCIDENT_RESPONSE_SECONDS_QY` and `INCIDENT_TRAVEL_TM_SECONDS_QY` to integer type
- Remove outliers from key numeric columns
- Create binary categorical variables for ladder assignment and long response times (>5 minutes)

### 2.1 Converting Datetime Columns

The date/time columns are currently stored as strings. We'll convert them to `datetime` objects using `pd.to_datetime()`, which will allow us to perform date arithmetic in the steps that follow.

In [ ]:
#set incident datetime, first on scene datetime, and incident close datetime to datetime data type
df.INCIDENT_DATETIME=pd.to_datetime(df.INCIDENT_DATETIME, format='%m/%d/%Y %I:%M:%S %p')
df.FIRST_ON_SCENE_DATETIME=pd.to_datetime(df.FIRST_ON_SCENE_DATETIME, format='%m/%d/%Y %I:%M:%S %p')
df.INCIDENT_CLOSE_DATETIME=pd.to_datetime(df.INCIDENT_CLOSE_DATETIME, format='%m/%d/%Y %I:%M:%S %p')

### 2.2 Computing Time on Scene

We want to know how long the fire department spent at each incident. Subtracting `FIRST_ON_SCENE_DATETIME` from `INCIDENT_CLOSE_DATETIME` gives us a `Timedelta` object; calling `.total_seconds()` converts it to a float. Let's verify the calculation on a single row before applying it to the full dataset.

In [ ]:
#calculate total time spent on scene. This example shows how the calculation works, just on the first row of data
diff=df.INCIDENT_CLOSE_DATETIME[0]-df.FIRST_ON_SCENE_DATETIME[0]
diff.total_seconds()

In [ ]:
#now let's do this on the entire dataset
time_on_scene=[]
for index, row in df.iterrows():
    arrive=row.FIRST_ON_SCENE_DATETIME
    close=row.INCIDENT_CLOSE_DATETIME
    diff=close-arrive
    time_on_scene.append(diff.total_seconds())
df['time_on_scene']=time_on_scene

df.head()

### 2.3 Extracting Date and Month

We'll add two convenience columns:
- **`date`**: just the calendar date (no time component), used for daily time series grouping
- **`month`**: the full month name, used for grouping and animation later in the workshop

In [ ]:
#lets create a column that has the incident date only - no time stamp
df['date']=[x.date() for x in df.INCIDENT_DATETIME]

In [ ]:
#let's make one additional column that only shows the month
df['month']=[d.strftime("%B") for d in df.date]

#lets inspect our new date columns
df.head()

### 2.4 Converting Numeric Columns

The response time columns (`INCIDENT_RESPONSE_SECONDS_QY` and `INCIDENT_TRAVEL_TM_SECONDS_QY`) are stored as strings and need to be converted to integers for numerical analysis. Let's try a direct conversion first.

In [ ]:
#set INCIDENT_RESPONSE_SECONDS_QY and INCIDENT_TRAVEL_TM_SECONDS_QY to integer type
df.INCIDENT_RESPONSE_SECONDS_QY=df.INCIDENT_RESPONSE_SECONDS_QY.astype(int)

The conversion failed because the column contains null (`NaN`) values — Python can't convert missing values to integers directly. Let's remove rows where response time is missing and try again.

In [ ]:
df=df.loc[~df.INCIDENT_RESPONSE_SECONDS_QY.isna()]
df.INCIDENT_RESPONSE_SECONDS_QY=df.INCIDENT_RESPONSE_SECONDS_QY.astype(int)

Still failing. The remaining values contain commas as thousands separators (e.g., `"1,234"`), which Python can't parse as a number. We'll use `.replace()` to strip the commas before converting. We'll apply the same fix to the travel time column at the same time.

In [ ]:
#change response time to integer
df.INCIDENT_RESPONSE_SECONDS_QY=[int(x.replace(',','')) for x in df.INCIDENT_RESPONSE_SECONDS_QY]

#remove null values from travel times if there are any) and convert to integers as well
df=df.loc[~df.INCIDENT_TRAVEL_TM_SECONDS_QY.isna()]
df.INCIDENT_TRAVEL_TM_SECONDS_QY=[int(x.replace(',','')) for x in df.INCIDENT_TRAVEL_TM_SECONDS_QY]

### 2.5 Identifying and Removing Outliers

With our numeric columns cleaned up, let's check for outliers using Seaborn box plots. Extreme outliers will distort our visualizations if left in.

In [ ]:
sns.boxplot(data=df,x='INCIDENT_RESPONSE_SECONDS_QY');

There are significant outliers in the response time distribution. These could be data entry errors or genuinely exceptional incidents — either way, they'll skew our charts.

A practical approach is to remove the top 1% of values using the **99th percentile** as a cutoff. This preserves the vast majority of the data while eliminating extreme values.

In [ ]:
#we'll call the 99th percentile q99
q99=np.nanquantile(df.INCIDENT_RESPONSE_SECONDS_QY,0.99)
#now let's remove anything above q99 - ie, take out the top 1%

df=df.loc[df.INCIDENT_RESPONSE_SECONDS_QY<=q99]
#then we'll review the box plot again
sns.boxplot(data=df,x='INCIDENT_RESPONSE_SECONDS_QY');

#### Travel Time

Now let's inspect the `INCIDENT_TRAVEL_TM_SECONDS_QY` distribution.

In [ ]:
#lets look at the travel times while we're at it
sns.boxplot(data=df,x='INCIDENT_TRAVEL_TM_SECONDS_QY');

Negative travel times are physically impossible and are likely data errors. Let's count them before deciding how to handle them.

In [ ]:
negative_travel_times=len(df.loc[df.INCIDENT_TRAVEL_TM_SECONDS_QY<0])
print(f'there are {negative_travel_times} rows with negative travel times')

In [ ]:
#remove the rows with negative travel times and review the box plot again
df=df.loc[df.INCIDENT_TRAVEL_TM_SECONDS_QY>0]
sns.boxplot(data=df,x='INCIDENT_TRAVEL_TM_SECONDS_QY');

#### Time on Scene

Now let's inspect the `time_on_scene` column we calculated earlier.

In [ ]:
sns.boxplot(data=df,x='time_on_scene');

The distribution is heavily skewed. We'll apply the same 99th percentile cutoff used for response time, and also remove any negative values.

In [ ]:
#we'll call the 99th percentile q99
q99=np.nanquantile(df.time_on_scene,0.99)
#now let's remove anything above q99 - ie, take out the top 1%

df=df.loc[(df.time_on_scene<=q99)&(df.time_on_scene>0)]
#then we'll review the box plot again
sns.boxplot(data=df,x='time_on_scene');

### 2.6 Creating Derived Variables

We'll create two binary categorical variables that will be useful for color-coding our charts:

- **`ladder_assigned`**: Whether a ladder truck was dispatched to the incident (`"Ladder Assigned"` / `"No Ladder Assigned"`)
- **`5min_wait`**: Whether the total response time exceeded 5 minutes (`">5 Mins"` / `"<5 Mins"`)

In [ ]:
#ladder assigned binary value
df['ladder_assigned']=['Ladder Assigned' if x>0 else 'No Ladder Assigned' for x in df['LADDERS_ASSIGNED_QUANTITY']]
#find responses longer than 5 minutes
df['5min_wait']=['>5 Mins' if x>5*60 else '<5 Mins' for x in df.INCIDENT_RESPONSE_SECONDS_QY]

df.head()

We'll also use the `ENGINES_ASSIGNED_QUANTITY` column later in the workshop. Let's check for missing values and remove those rows if there are only a small number.

In [ ]:
df.ENGINES_ASSIGNED_QUANTITY.isna()
missing_engines_assigned=len(df.loc[df.ENGINES_ASSIGNED_QUANTITY.isna()])
print(f'there are {missing_engines_assigned} rows with no data in the engines assigned column')

In [ ]:
#remove rows with missing values
df=df.loc[~df.ENGINES_ASSIGNED_QUANTITY.isna()]

## 3. Bar Charts

Bar charts are ideal for comparing counts or aggregated values across categories. We'll build a chart showing fire incident counts by borough, starting simple and progressively adding more information.

First, we need to aggregate the data. We'll use `groupby()` to count incidents per borough and sort from highest to lowest.

In [ ]:
# bar plots - first, let's make a dataframe that is grouped by borough.
df_bar=df.groupby(['INCIDENT_BOROUGH']).INCIDENT_BOROUGH.count().reset_index(name='incident_count')
#now let's sort the dataframe from highest to lowest count
df_bar=df_bar.sort_values(by='incident_count', ascending=False)
df_bar

In [ ]:
fig = px.bar(df_bar, x='INCIDENT_BOROUGH', y='incident_count')
fig.show()

Plotly automatically generates an interactive chart — hover over any bar to see the exact count. Next, let's encode an additional variable using color to make the chart more informative.

### Stacked and Grouped Bar Charts

Adding a `color` parameter encodes a second variable visually. By default, Plotly stacks the bars, showing the breakdown of ladder assignments within each borough.

In [ ]:
# let's regroup our data, but this time add the ladder_assigned variable. This way we'll see the counts of incidents by borough, broken down by which one's had a ladder assigned.
df_bar=df.groupby(['INCIDENT_BOROUGH', 'ladder_assigned']).INCIDENT_BOROUGH.count().reset_index(name='incident_count')
#now let's sort the dataframe from highest to lowest count
df_bar=df_bar.sort_values(by='incident_count', ascending=False)
df_bar

fig = px.bar(df_bar, x='INCIDENT_BOROUGH', y='incident_count', color='ladder_assigned')
fig.show()

A side-by-side grouped layout can make it easier to compare individual categories directly. Switch to this mode with `barmode='group'`.

In [ ]:
#we can also make a grouped version rather than stacked
fig = px.bar(df_bar, x='INCIDENT_BOROUGH', y='incident_count', color='ladder_assigned', barmode='group')
fig.show()

### Customizing Labels and Layout

Raw column names make for poor chart labels. The `labels` parameter lets us rename legend entries, and `update_layout()` lets us set axis titles, a chart title, and hover styling.

In [ ]:
#lets adjust some of the labels to make this more professional
fig = px.bar(df_bar, x='INCIDENT_BOROUGH', y='incident_count', color='ladder_assigned', labels={"ladder_assigned": "Ladder Assignment"})
#use .update_laout() method to change some of the formatting options
fig.update_layout(
    xaxis_title='', #remove x-axis label because it's obvious
    yaxis_title='Incident Count',
    title='Fire Incidents by Borough: July-December 2024',
    hoverlabel=dict(
        bgcolor="white", # Set the background color
        font_size=12,
        font_family="Arial"
    )
)
fig.show()

## 4. Histograms and Box Plots

Histograms and box plots are the primary tools for exploring distributions. Let's visualize `INCIDENT_RESPONSE_SECONDS_QY` — the time in seconds from dispatch to first unit arrival on scene.

In [ ]:
fig = px.histogram(df, x='INCIDENT_RESPONSE_SECONDS_QY')
fig.show()

Adding a `color` parameter overlays the distributions for each category on the same histogram, making it easy to compare their shapes.

In [ ]:
#we can make it more interesting by adding in a categorical variable
fig = px.histogram(df, x='INCIDENT_RESPONSE_SECONDS_QY', color='ladder_assigned')
fig.show()

### Box Plots

Box plots display more distributional detail than histograms: median, interquartile range, and individual outliers are all visible at once.

In [ ]:
#lets try a box plot
fig = px.box(df, x='INCIDENT_RESPONSE_SECONDS_QY')
fig.show()

Passing a `color` variable produces one box per group, allowing direct comparison of response time distributions across all five boroughs.

In [ ]:
#lets make it vertical and look at the distribution by borough
fig = px.box(df, y='INCIDENT_RESPONSE_SECONDS_QY', color='INCIDENT_BOROUGH')
fig.show()

## 5. Scatter Plots and Bubble Charts

Scatter plots reveal relationships between two continuous variables. Let's examine whether there's any relationship between time on scene and travel time.

Since the full dataset has ~200,000 rows, we'll sample 1% using `.sample(frac=0.01)` to keep rendering fast without losing the overall pattern.

In [ ]:
fig = px.scatter(df.sample(frac=0.01), x='time_on_scene', y='INCIDENT_TRAVEL_TM_SECONDS_QY')
fig.show()

No strong relationship is apparent between these two variables. Let's add a color dimension to see if any pattern emerges when we distinguish between incidents with and without a ladder assigned.

In [ ]:
fig = px.scatter(df.sample(frac=0.01), x='time_on_scene', y='INCIDENT_TRAVEL_TM_SECONDS_QY', color='ladder_assigned')
fig.show()

### Bubble Charts

A bubble chart extends a scatter plot by encoding a third variable as the **size** of each point. Here we'll map `INCIDENT_RESPONSE_SECONDS_QY` to point size, adding a fourth dimension to the visualization.

In [ ]:
#lets throw in a third variable to make this a bubble chart. We'll map the incident response seconds to the size of the points on the scatterplot

fig = px.scatter(df.sample(frac=0.01), x='time_on_scene', y='INCIDENT_TRAVEL_TM_SECONDS_QY', color='ladder_assigned', size='INCIDENT_RESPONSE_SECONDS_QY')
fig.show()

## 6. Line Charts

Line charts are well-suited for visualizing data over time. Let's examine how the daily number of fire incidents varied across the five months in our dataset.

First, we'll group the data by date and count incidents per day.

In [ ]:
df_line=df.groupby(['date']).STARFIRE_INCIDENT_ID.count().reset_index(name='incident_count')
df_line

In [ ]:
fig = px.line(df_line,x='date',y='incident_count')
fig.show()

### Adding a Rolling Average

Day-to-day counts can be noisy. A rolling (moving) average smooths the trend, making long-term patterns easier to see. We'll use Pandas `.rolling()` to compute a 10-day moving average, then plot it alongside the raw daily counts.

In [ ]:
#lets add a rolling average! We'll use the .rolling() method in pandas
df_line['10day_moving_avg']=df_line.incident_count.rolling(10).mean()
df_line

In [ ]:
fig = px.line(df_line,x='date',y=['incident_count', '10day_moving_avg'])
fig.show()

### Breaking Down by Category

Adding a `color` parameter produces one line per borough, letting us compare trends across the city simultaneously.

In [ ]:
#lets break it down by borough
df_line2=df.groupby(['date', 'INCIDENT_BOROUGH']).STARFIRE_INCIDENT_ID.count().reset_index(name='incident_count')
fig = px.line(df_line2,x='date',y='incident_count', color='INCIDENT_BOROUGH')
fig.show()

### Unified Hover Mode

With five lines, hovering over each one individually is tedious. Setting `hovermode="x unified"` consolidates all values at the same x-position into a single tooltip.

In [ ]:
#it's a lot of work to hover over all 5 lines. We can "unify" the hover across all points on the x-axis by changing the "hovermode" to "x unified". Give it a shot
fig = px.line(df_line2,x='date',y='incident_count', color='INCIDENT_BOROUGH')
fig.update_layout(hovermode="x unified")
fig.show()

## 7. Animation

Plotly supports animation by stepping through the values of a variable frame by frame. This is useful for showing how a relationship changes over time or across categories.

We'll re-use the bubble chart from earlier and animate it across months. The `animation_frame` parameter specifies which variable to animate over, and `category_orders` ensures the months appear in the correct sequence.

In [ ]:
month_order=['August', 'September', 'October', 'November', 'December']
fig = px.scatter(df.sample(frac=0.01), x='time_on_scene', y='INCIDENT_TRAVEL_TM_SECONDS_QY', color='ladder_assigned', size='INCIDENT_RESPONSE_SECONDS_QY',
                animation_frame='month', 
                 category_orders={'month': month_order}) # This controls the order)
fig.update_layout(
    xaxis_title='Time on Scene', #remove x-axis label because it's obvious
    yaxis_title='Travel Time',
    title='Time on Scene vs Travel Time by Month',
    hoverlabel=dict(
        bgcolor="white", # Set the background color
        font_size=12,
        font_family="Arial"
    )
)
fig.show()

## 8. Sankey Diagrams

Sankey diagrams visualize flows between a sequence of categories. They are built using `plotly.graph_objects` (`go`) rather than `plotly.express`, and require more data preparation than the chart types we've covered so far.

The key components of a Sankey diagram are:

| Component | Description |
|-----------|-------------|
| **Labels** | The name of each node in the diagram |
| **Sources** | The index (in the labels list) from which flow originates |
| **Targets** | The index (in the labels list) that receives the flow |
| **Values** | The quantity flowing from each source to each target |

We'll trace how incidents flow through three variables: **Borough → Ladder Assignment → Response Time Category**.

### Preparing the Data

We need to build three parallel lists — `source`, `target`, and `value` — that describe every connection in the diagram. We loop through each adjacent pair of variable columns and count how many incidents flow between each combination of categories.

In [ ]:
#prepare the data for the diagram
set1=df['INCIDENT_BOROUGH'].unique().tolist()
set2=df['ladder_assigned'].unique().tolist()
set3=df['5min_wait'].unique().tolist()
set_dict={'INCIDENT_BOROUGH':set1,
         'ladder_assigned':set2,
         '5min_wait': set3}
labels =set1+set2+set3

source=[]
target=[]
value=[]
sets=[k for k in set_dict]
for i, l in enumerate(sets):
    #stop before last set - no more connections from there
    if l!=sets[-1]:
        sources=df[l].unique()
        for s in sources:
            #source.append(labels.index(s))
            targets=df.loc[df[l]==s][sets[i+1]].unique().tolist()
            for t in targets:
                source.append(labels.index(s))
                target.append(labels.index(t))
                val=len(df.loc[(df[l]==s)&(df[sets[i+1]]==t)])
                value.append(val)

### Building the Figure

With the data prepared, we pass our `labels`, `source`, `target`, and `value` lists to `go.Sankey()`. The `node` dictionary controls the appearance of each node, and the `link` dictionary defines the connections between them.

In [ ]:
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15,
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = labels,
      color = "blue",
      hovertemplate='%{label}: %{value}<extra></extra>'
    ),
    link = dict(
      source = source, 
      target = target,
      value = value,
      hovertemplate='%{source.label} and %{target.label}: %{value}<extra></extra>'
  ))])

fig.update_layout(title="<span style='font-size:30px;'><b>Fire Response by Borough</b></span>")
#fig.update_layout(title_text='Fire Response by Borough', title_font=36)
fig.show()

For more examples and customization options, see the [Plotly Sankey Diagram documentation](https://plotly.com/python/sankey-diagram/).

## 9. Exporting Charts

Plotly charts can be saved as standalone HTML files that preserve full interactivity — no Python or Jupyter required to view them. Use the `.write_html()` method on any figure object.

In [ ]:
fig.write_html("sankey_example.html")